In [1]:
import requests
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Load preprocessed dataset and train Decision Tree model once
df = pd.read_csv('preprocessed_airquality_data.csv')

feature_cols = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3']
target_col = 'AQI_Category'

df = df.dropna(subset=[target_col])
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col])

X = df[feature_cols]
y = df[target_col]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_scaled, y)

# Functions to fetch real-time data
def get_weather_data(city, api_key):
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return {
            'temperature': data['main']['temp'],
            'humidity': data['main']['humidity'],
            'wind_speed': data['wind']['speed']
        }
    else:
        print("Error fetching weather data")
        return None

def get_aqi_data(city, token):
    url = f'https://api.waqi.info/feed/{city}/?token={token}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['status'] == 'ok':
            aqi = data['data']['iaqi']
            pollutants = {}
            # Extract pollutants if available, default 0 otherwise
            pollutants['PM2.5'] = aqi.get('pm25', {'v': 0})['v']
            pollutants['PM10'] = aqi.get('pm10', {'v': 0})['v']
            pollutants['NO2'] = aqi.get('no2', {'v': 0})['v']
            pollutants['CO'] = aqi.get('co', {'v': 0})['v']
            pollutants['O3'] = aqi.get('o3', {'v': 0})['v']
            return pollutants
        else:
            print("API returned bad status")
            return None
    else:
        print("Error fetching AQI data")
        return None

# Integrate everything and predict AQI category for a city
def predict_aqi_category(city, weather_api_key, aqi_api_token):
    pollutants = get_aqi_data(city, aqi_api_token)
    if not pollutants:
        return "No pollutant data available"
    
    # Prepare input for model
    input_df = pd.DataFrame([pollutants])
    # Scale features like training set
    input_scaled = scaler.transform(input_df)
    
    pred = model.predict(input_scaled)
    category = le.inverse_transform(pred)[0]
    return category

# Example usage
openweather_api_key = 'YOUR_OPENWEATHERMAP_API_KEY'
aqicn_token = 'YOUR_AQICN_API_TOKEN'
city_name = 'Delhi'

print(f"Predicting AQI Category for {city_name}...")
predicted_category = predict_aqi_category(city_name, openweather_api_key, aqicn_token)
print(f"Predicted AQI Category: {predicted_category}")
 

Predicting AQI Category for Delhi...
API returned bad status
Predicted AQI Category: No pollutant data available
